# tiny-ton GEMM Benchmark: Road to cuBLAS

This notebook benchmarks **tiny-ton's Python DSL kernels** against cuBLAS and maps the journey from a naive matvec to full GEMM performance.

| Stage | Kernel | What it adds |
|-------|--------|---------------|
| **0** | Naive matvec | One block per row, all K in a single load |
| **1** | Tiled matvec | K tiled with compile-time loop, arbitrary K |
| **2** | Shared-mem matvec | Load x once into shared memory, 2 rows per block |
| **3** | Full GEMM | Nested compile-time loops over cols × K tiles |
| **∞** | cuBLAS | 2D tiles, shared mem for A+B, tensor cores, async |

Each level exposes what the compiler needs next.


## 1. One-time setup — run in your Jetson terminal

`sudo` needs an interactive terminal, so paste these commands once via `ssh jetson`.

**Step 1 — Install LLVM 18 + build tools:**
```bash
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | sudo tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
echo "deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main" | sudo tee /etc/apt/sources.list.d/llvm-18.list
sudo apt-get update -qq
sudo apt-get install -y libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build
pip install pybind11
```

**Step 2 — Clone and build tiny-ton (~3 min):**
```bash
git clone https://github.com/ganeshnj/tiny-ton.git ~/tiny-ton
cd ~/tiny-ton
cmake -G Ninja -S . -B build \\
  -DCMAKE_BUILD_TYPE=Release \\
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \\
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \\
  -DTTN_ENABLE_CUDA=ON \\
  -DTTN_ENABLE_PYTHON=ON \\
  -DCUDAToolkit_ROOT=/usr/local/cuda \\
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
```

When both steps finish, run the check cell below then continue.

In [1]:
import shutil, os

checks = {
    'mlir-opt-18':    shutil.which('mlir-opt-18') is not None,
    'cmake':          shutil.which('cmake') is not None,
    'ninja':          shutil.which('ninja') is not None,
    'tiny-ton build': os.path.exists(os.path.expanduser('~/tiny-ton/build/bindings')),
}
all_ok = True
for name, ok in checks.items():
    tag = 'OK' if ok else 'MISSING  <- run Step 1/2 above first'
    print(f'  {name:20s}  {tag}')
    all_ok = all_ok and ok

if not all_ok:
    raise SystemExit('Setup incomplete.')
print('\nAll checks passed — continue to cell 2.')


  mlir-opt-18           OK
  cmake                 OK
  ninja                 OK
  tiny-ton build        OK

All checks passed — continue to cell 2.


In [2]:
import sys, os, time
import numpy as np

TINY_TON_ROOT = os.path.expanduser('~/tiny-ton')
sys.path.insert(0, os.path.join(TINY_TON_ROOT, 'build', 'bindings'))
sys.path.insert(0, os.path.join(TINY_TON_ROOT, 'python'))

os.environ['TTN_SM_VERSION'] = 'sm_87'   # Jetson Orin Nano — Ampere

import _tiny_ton_core as core
import tiny_ton as tt

print(f'has_cuda()   = {core.has_cuda()}')
print(f'SM version   = {os.environ["TTN_SM_VERSION"]}')
print(f'NumPy        = {np.__version__}')


has_cuda()   = True
SM version   = sm_87
NumPy        = 1.21.5


In [3]:
# ── Timing helper ────────────────────────────────────────────────────────────
# tiny-ton copies numpy arrays to/from device synchronously on each call,
# so wall-clock time captures the full H2D + kernel + D2H roundtrip.

def tflops(m, n, k, ms):
    """TFLOPS for a M×K @ K×N matmul given elapsed milliseconds."""
    return 2.0 * m * n * k / (ms * 1e-3) / 1e12

def benchmark_ttn(kernel_fn, grid, args, warmup=10, reps=50):
    """
    Warm up the JIT, then time `reps` consecutive calls.
    Returns average milliseconds per call.
    """
    for _ in range(warmup):
        kernel_fn[grid](*args)
    t0 = time.perf_counter()
    for _ in range(reps):
        kernel_fn[grid](*args)
    return (time.perf_counter() - t0) / reps * 1000.0

print('Timing helpers ready.')


Timing helpers ready.


## 3. cuBLAS Reference Numbers

From `cublas_matmul.ipynb` (already run on this Jetson Orin Nano).  
These are the targets every tiny-ton kernel level is measured against.

| Shape | FP32 SGEMM | FP16 HGEMM (tensor cores) |
|-------|-----------|---------------------------|
| 64³   | ~0.005 TF  | ~0.02 TF                  |
| 128³  | ~0.03 TF   | ~0.12 TF                  |
| 256³  | ~0.2 TF    | ~0.7 TF                   |
| 512³  | ~0.6 TF    | ~2.5 TF                   |
| 1024³ | ~1.2 TF    | ~5 TF                     |
| 2048³ | ~1.8 TF    | ~9 TF                     |
| 4096³ | ~2.0 TF    | ~12 TF                    |

**Goal**: match the FP32 SGEMM column first (Phase A+B), then FP16 with tensor cores (Phase C).


## 4. Level 0 — Naive Matvec

```
Grid: (M,)   — one block per output row
Block: 64 threads
K limit: ≤ 64   (one load per thread, no loop)
```

Each block loads the entire weight row `W[row, :]` and the input vector `x[:]`
in a single instruction, multiplies element-wise, then reduces with a warp shuffle.
Works for any M, but K is hard-capped at 64 (block size).

**Bottleneck**: K ≤ 64 — too small for any real model layer.


In [4]:
# ── Level 0: naive matvec ─────────────────────────────────────────────────────
# C[M] = A[M,K] @ B[K]   (B is a vector, not a matrix)
# K ≤ 64: all K elements loaded in one shot, no loop.

@tt.jit
def naive_matvec(A_ptr, x_ptr, y_ptr, K: tt.constexpr):
    row = tt.program_id(0)
    tid = tt.arange(0, K)
    w   = tt.load(A_ptr + row * K + tid)
    x   = tt.load(x_ptr + tid)
    tt.store(y_ptr + row, tt.reduce_sum(w * x))


# ── Correctness ───────────────────────────────────────────────────────────────
np.random.seed(0)
M_l0, K_l0 = 64, 64
A0 = np.random.randn(M_l0, K_l0).astype(np.float32)
x0 = np.random.randn(K_l0).astype(np.float32)
y0 = np.zeros(M_l0, dtype=np.float32)
naive_matvec[(M_l0,)](A0.copy(), x0.copy(), y0, K_l0)
err = np.max(np.abs(y0 - A0 @ x0))
print(f'Level 0 correctness (M={M_l0}, K={K_l0}): max_err={err:.2e}')
assert err < 1e-4

# ── Benchmark ────────────────────────────────────────────────────────────────
l0_best_tf = 0.0
print(f'\n{"Shape (MxK)":>14s}  {"ms":>8s}  {"GFLOPS":>8s}')
print('-' * 36)
for M in [64, 128, 256, 512, 1024, 2048]:
    K = 64
    A = np.random.randn(M, K).astype(np.float32)
    x = np.random.randn(K).astype(np.float32)
    y = np.zeros(M, dtype=np.float32)
    ms = benchmark_ttn(naive_matvec, grid=(M,), args=(A, x, y, K))
    gf = tflops(M, 1, K, ms) * 1000   # GFLOPS
    l0_best_tf = max(l0_best_tf, gf / 1000)
    print(f'{f"{M}x{K}":>14s}  {ms:8.4f}  {gf:8.3f}')
print(f'\nNote: K capped at 64. Timing includes H2D+kernel+D2H per call.')


Level 0 correctness (M=64, K=64): max_err=1.91e-06

   Shape (MxK)        ms    GFLOPS
------------------------------------
         64x64  119.3572     0.000
        128x64  116.7384     0.000
        256x64  119.6515     0.000
        512x64  117.0694     0.001
       1024x64  122.4086     0.001
       2048x64  115.6218     0.002

Note: K capped at 64. Timing includes H2D+kernel+D2H per call.


**What's limiting Level 0:**
- K ≤ 64 (block size).  A single GPT-2 attention projection has K=768.
- Only a matvec (B is a vector), not a GEMM.  M×N output requires N separate launches.

**What Level 1 adds:** compile-time for-loop over K tiles, removing the K limit.


## 5. Level 1 — Tiled Matvec

```
Grid: (M,)   — one block per output row
Block: TILE_K threads
K limit: none — tiled via compile-time unrolled for loop
```

The JIT unrolls `for k0 in range(0, K, TILE_K)` at compile time because both
`K` and `TILE_K` are `tt.constexpr`.  The accumulator `acc` chains across iterations
via SSA value bindings — no phi nodes in the IR.

**Bottleneck**: still a matvec (N=1), so no column parallelism.  Every block
re-reads `x` from global memory independently — no data reuse.


In [5]:
# ── Level 1: tiled matvec ─────────────────────────────────────────────────────
# C[M] = A[M,K] @ B[K]  — K tiled with compile-time loop

@tt.jit
def tiled_dot_kernel(A_ptr, B_ptr, C_ptr, K: tt.constexpr, TILE_K: tt.constexpr):
    row = tt.program_id(0)
    tid = tt.arange(0, TILE_K)
    acc = 0.0
    for k0 in range(0, K, TILE_K):
        a_tile = tt.load(A_ptr + row * K + k0 + tid)
        b_tile = tt.load(B_ptr + k0 + tid)
        acc = acc + tt.reduce_sum(a_tile * b_tile)
    tt.store(C_ptr + row, acc)


# ── Correctness ───────────────────────────────────────────────────────────────
np.random.seed(1)
M_l1, K_l1, TK = 64, 256, 64
A1 = np.random.randn(M_l1, K_l1).astype(np.float32)
x1 = np.random.randn(K_l1).astype(np.float32)
y1 = np.zeros(M_l1, dtype=np.float32)
tiled_dot_kernel[(M_l1,)](A1.copy(), x1.copy(), y1, K_l1, TK)
err = np.max(np.abs(y1 - A1 @ x1))
print(f'Level 1 correctness (M={M_l1}, K={K_l1}, TILE_K={TK}): max_err={err:.2e}')
assert err < 1e-3

# ── Benchmark ────────────────────────────────────────────────────────────────
l1_best_tf = 0.0
print(f'\n{"Shape":>14s}  {"TILE_K":>6s}  {"ms":>8s}  {"GFLOPS":>8s}')
print('-' * 44)
for M, K, TILE_K in [
    (512,   256, 64),
    (512,   512, 64),
    (1024,  256, 64),
    (2048,  256, 64),
    (4096,  256, 64),
]:
    A = np.random.randn(M, K).astype(np.float32)
    x = np.random.randn(K).astype(np.float32)
    y = np.zeros(M, dtype=np.float32)
    ms = benchmark_ttn(tiled_dot_kernel, grid=(M,), args=(A, x, y, K, TILE_K))
    gf = tflops(M, 1, K, ms) * 1000
    l1_best_tf = max(l1_best_tf, gf / 1000)
    print(f'{f"{M}x{K}":>14s}  {TILE_K:>6d}  {ms:8.4f}  {gf:8.3f}')


Level 1 correctness (M=64, K=256, TILE_K=64): max_err=2.64e+01


AssertionError: 

**What's limiting Level 1:**
- N=1 only.  Computing a full M×N output requires N separate calls — huge launch overhead.
- Each block loads `x` independently.  For M=4096 blocks, `x` is loaded 4096 times.

**What Level 2 adds:** shared memory to load `x` once and share it across rows in the same block.


## 6. Level 2 — Shared Memory Matvec

```
Grid: (M/2,)  — one block computes 2 output rows
Block: TILE_K threads
Shared mem: x loaded once, reused for both rows
```

The input vector `x` is stored into shared memory once per block, then reused
for two dot products.  This halves the number of kernel blocks and doubles
the global-memory reads that are eliminated by reuse.

**Bottleneck**: still matvec (N=1), and we only share across 2 rows.  For real GEMM
we need to tile both A and B into shared memory and compute a full output tile.


In [ ]:
# ── Level 2: shared memory matvec (2 rows per block) ─────────────────────────
# Each block loads x into shared memory once, then dots with 2 consecutive rows.

@tt.jit
def shmem_matvec(W_ptr, x_ptr, y_ptr, in_features: tt.constexpr, BLOCK: tt.constexpr):
    pid = tt.program_id(0)          # block index → rows pid*2 and pid*2+1
    tid = tt.arange(0, BLOCK)
    mask = tid < in_features
    # Load x into shared memory — one global load, two dot products benefit
    x_val = tt.load(x_ptr + tid, mask=mask)
    tt.shared_store(tid, x_val)
    tt.sync()
    x_sh = tt.shared_load(tid)
    # Two rows per block
    w0 = tt.load(W_ptr + (pid * 2)     * in_features + tid, mask=mask)
    w1 = tt.load(W_ptr + (pid * 2 + 1) * in_features + tid, mask=mask)
    tt.store(y_ptr + pid * 2,     tt.reduce_sum(w0 * x_sh))
    tt.store(y_ptr + pid * 2 + 1, tt.reduce_sum(w1 * x_sh))


# ── Correctness ───────────────────────────────────────────────────────────────
np.random.seed(2)
M_l2, K_l2 = 64, 64
W2 = np.random.randn(M_l2, K_l2).astype(np.float32)
x2 = np.random.randn(K_l2).astype(np.float32)
y2 = np.zeros(M_l2, dtype=np.float32)
shmem_matvec[(M_l2 // 2,)](W2.copy(), x2.copy(), y2, K_l2, K_l2)
err = np.max(np.abs(y2 - W2 @ x2))
print(f'Level 2 correctness (M={M_l2}, K={K_l2}): max_err={err:.2e}')
assert err < 1e-4

# ── Benchmark: compare Level 1 vs Level 2 at same sizes ──────────────────────
l2_best_tf = 0.0
l2_results = {}

print(f'\n{"Shape":>14s}  {"Level 1 (ms)":>13s}  {"Level 2 (ms)":>13s}  {"Speedup":>7s}')
print('-' * 58)
for M, K in [(256, 64), (512, 64), (1024, 64), (2048, 64)]:
    W = np.random.randn(M, K).astype(np.float32)
    x = np.random.randn(K).astype(np.float32)
    y = np.zeros(M, dtype=np.float32)

    ms_l1 = benchmark_ttn(tiled_dot_kernel,
                            grid=(M,), args=(W, x, y, K, K))
    ms_l2 = benchmark_ttn(shmem_matvec,
                            grid=(M // 2,), args=(W, x, y, K, K))
    speedup = ms_l1 / ms_l2
    gf = tflops(M, 1, K, ms_l2) * 1000
    shape = f'{M}x{K}'
    l2_results[(M, K)] = gf / 1000
    print(f'{shape:>14s}  {ms_l1:13.4f}  {ms_l2:13.4f}  {speedup:7.2f}x')
    l2_best_tf = max(l2_best_tf, tflops(M, 1, K, ms_l2))


**What's limiting Level 2:**
- Still a matvec — output is a vector, not a matrix.
- Shared memory is loaded from a 1D offset; we can't independently tile A and B for GEMM.
- No column loop — each call produces one output column.

**What Level 3 adds:** nested compile-time loops over output columns and K tiles,
enabling full M×N GEMM in a single kernel call.


## 7. Level 3 — Full GEMM with Compile-Time Loops

```
Grid: (M,)    — one block per output row
Block: TILE_K threads
K limit: none — tiled with compile-time loop
N limit: must be tt.constexpr — unrolled at compile time
```

The JIT compiler unrolls both `for col in range(N)` and `for k0 in range(0,K,TILE_K)`
at compile time (all bounds are `tt.constexpr`).
One block computes an entire output row `C[row, :]` — all N columns.

**Bottleneck**: N and K must be compile-time constants.  Large N/K causes IR explosion.
For N=256, TILE_K=16: 256×16 = 4096 unrolled loop bodies.
No 2D output tiling — all output columns are in one block → no inter-block parallelism on N.


In [ ]:
# ── Level 3: full GEMM with nested compile-time loops ────────────────────────
# C[M,N] = A[M,K] @ B[K,N]
# One block per output row.  Both col-loop (N) and K-loop are constexpr-unrolled.

@tt.jit
def tiled_matmul_kernel(A_ptr, B_ptr, C_ptr,
                        K: tt.constexpr, N: tt.constexpr,
                        TILE_K: tt.constexpr):
    row = tt.program_id(0)
    tid = tt.arange(0, TILE_K)
    for col in range(N):
        acc = 0.0
        for k0 in range(0, K, TILE_K):
            # A[row, k0:k0+TILE_K]
            a_tile = tt.load(A_ptr + row * K + k0 + tid)
            # B[k0:k0+TILE_K, col]  (row-major: B[k,col] = B_ptr[k*N + col])
            b_tile = tt.load(B_ptr + k0 * N + col + tid * N)
            acc = acc + tt.reduce_sum(a_tile * b_tile)
        tt.store(C_ptr + row * N + col, acc)


# ── Correctness ───────────────────────────────────────────────────────────────
np.random.seed(3)
for M, K, N, TK in [(4, 8, 4, 4), (8, 16, 8, 8), (16, 32, 16, 16)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    tiled_matmul_kernel[(M,)](A.copy(), B.copy(), C, K, N, TK)
    err = np.max(np.abs(C - A @ B))
    print(f'Level 3 M={M} K={K} N={N} TILE_K={TK}: max_err={err:.2e}'
          + (' ✓' if err < 1e-3 else ' ✗'))

# ── Benchmark  ────────────────────────────────────────────────────────────────
# N must be a Python literal (constexpr); we keep sizes small to avoid IR explosion.
# Each size shows:  tiny-ton TFLOPS  vs  cuBLAS TFLOPS at same shape.
l3_results = {}

print(f'\n{"Shape (MxNxK)":>16s}  {"tiny-ton ms":>12s}  {"tiny-ton TF":>12s}  {"cuBLAS TF":>10s}  {"Gap":>6s}')
print('-' * 70)

configs = [
    (16,  16,  16,  16),
    (32,  32,  32,  16),
    (64,  64,  64,  16),
    (128, 128, 128, 16),
]

for M, N, K, TILE_K in configs:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    ms_ttn = benchmark_ttn(tiled_matmul_kernel,
                            warmup=5, reps=20,
                            grid=(M,), args=(A, B, C, K, N, TILE_K))
    tf_ttn = tflops(M, N, K, ms_ttn)
    tf_cub = cublas_results.get((M, N, K), float('nan'))
    gap    = tf_cub / tf_ttn if tf_ttn > 0 else float('inf')
    shape  = f'{M}x{N}x{K}'
    l3_results[(M, N, K)] = tf_ttn
    print(f'{shape:>16s}  {ms_ttn:12.4f}  {tf_ttn:12.6f}  {tf_cub:10.4f}  {gap:6.1f}x')

print('\n* cuBLAS TFLOPS measured at same shape.  Gap = cuBLAS / tiny-ton.')

## 8. Gap Analysis

Summary: every tiny-ton level vs cuBLAS at the same and at peak (4096³) shapes.

In [ ]:
# ── Gap analysis — all levels vs cuBLAS reference ────────────────────────────
# cuBLAS numbers from cublas_matmul.ipynb (Jetson Orin Nano, Ampere sm_87)

cublas_fp32 = {   # TFLOPS at square sizes
    64:   0.005,
    128:  0.03,
    256:  0.2,
    512:  0.6,
    1024: 1.2,
    2048: 1.8,
    4096: 2.0,
}
cublas_fp16_peak = 12.0  # TFLOPS (tensor cores, 4096³)

print('='*72)
print(' tiny-ton vs cuBLAS — Jetson Orin Nano (Ampere sm_87, FP32)')
print('='*72)
print(f'{"Stage":28s}  {"Shape":12s}  {"tiny-ton TF":>12s}  {"cuBLAS TF":>10s}  {"Gap":>6s}')
print('-'*72)

rows = []

# Level 0 — naive matvec (M=2048, K=64, N=1)
if 'l0_best_tf' in dir():
    rows.append(('L0  naive matvec',  '2048×64 (N=1)', l0_best_tf,   None, 'K≤64 only'))

# Level 1 — tiled matvec (M=4096, K=256, N=1)
if 'l1_best_tf' in dir():
    rows.append(('L1  tiled matvec',  '4096×256 (N=1)', l1_best_tf,  None, 'matvec, N=1'))

# Level 2 — shmem matvec (M=2048, K=64)
if 'l2_best_tf' in dir():
    rows.append(('L2  shmem matvec',  '2048×64 (N=1)', l2_best_tf,   None, '2 rows/block'))

# Level 3 — full GEMM — best measured
if 'l3_results' in dir() and l3_results:
    best_shape = max(l3_results, key=l3_results.get)
    best_tf    = l3_results[best_shape]
    sz         = best_shape[0]
    cub        = cublas_fp32.get(sz)
    gap        = f'{cub/best_tf:.0f}×' if cub and best_tf > 0 else 'n/a'
    rows.append(('L3  full GEMM',
                 f'{sz}³',
                 best_tf,
                 cub,
                 gap))

for stage, shape, ttn_tf, cub_tf, note in rows:
    cub_str = f'{cub_tf:.4f}' if cub_tf else '  —  '
    gap_str = note if isinstance(note, str) and '×' in note else note
    print(f'{stage:28s}  {shape:12s}  {ttn_tf:12.6f}  {cub_str:>10s}  {gap_str}')

print()
if 'l3_results' in dir() and l3_results:
    best_tf = max(l3_results.values())
    if best_tf > 0:
        print(f'Gap to cuBLAS FP32 peak (4096³):  {cublas_fp32[4096]/best_tf:,.0f}×')
        print(f'Gap to cuBLAS FP16 peak (4096³):  {cublas_fp16_peak/best_tf:,.0f}×')

print()
print('Main gap sources:')
print('  1. H2D + D2H transfer on every call (tiny-ton copies numpy each call)')
print('  2. Problem size: 128³ has 512× fewer FLOPs than 4096³')
print('  3. No 2D output tiling (all columns in one block, no inter-block N parallelism)')


## 9. Roadmap to cuBLAS Performance

The gap breaks into three categories of improvements.  Each row shows what
compiler/runtime feature unlocks it and what speedup it delivers.

---

### Phase A — Remove size limits (compiler changes)

| Step | What to add to tiny-ton | Impact |
|------|--------------------------|--------|
| A1 | **`program_id(1)`** — 2D grid launch | Blocks tile the output in both M and N; removes the N=constexpr constraint |
| A2 | **`scf.for` runtime loop** | K loop is a real IR loop, not compile-time unrolled; eliminates IR explosion |
| A3 | **Shared memory for A tile** | Each block loads a TILE_M×TILE_K strip of A once; reuse across N output columns |
| A4 | **Shared memory for B tile** | Each block loads a TILE_K×TILE_N strip of B once; reuse across M output rows |

After Phase A, the kernel structure is identical to a standard CUDA tiled GEMM:
```python
@tt.jit
def tiled_gemm(A_ptr, B_ptr, C_ptr,
               M, N, K,
               TILE_M: tt.constexpr, TILE_N: tt.constexpr, TILE_K: tt.constexpr):
    pid_m = tt.program_id(0)    # ← needs program_id(1)
    pid_n = tt.program_id(1)
    acc   = 0.0
    for k0 in range(0, K, TILE_K):   # ← needs scf.for
        a_tile = tt.load(...)          # TILE_M × TILE_K from shared mem
        b_tile = tt.load(...)          # TILE_K × TILE_N from shared mem
        acc    = acc + a_tile @ b_tile  # ← needs 2D tile dot
    tt.store(...)
```

Expected improvement after Phase A: **10–50×** — now compute-bound at large sizes.

---

### Phase B — Memory efficiency (microarchitecture)

| Step | What to add | Impact |
|------|-------------|--------|
| B1 | **Register tiling** — each thread computes a TILE_T×TILE_T sub-block | Reduces shared-mem pressure; more compute per load |
| B2 | **Vectorized loads** — `LDG.128` (load 4 floats in one instruction) | 4× memory bandwidth per instruction |
| B3 | **Shared memory swizzling** — permute store offsets to avoid bank conflicts | Eliminates 4-way bank conflicts; near-peak bandwidth |
| B4 | **`cp.async`** — async H2D copy (Ampere) | Overlaps compute and memory; hides memory latency |

Expected improvement after Phase B: **2–5×** on top of Phase A.

---

### Phase C — Tensor cores (hardware multiply-accumulate)

| Step | What to add | Impact |
|------|-------------|--------|
| C1 | **`mma.sync.m16n8k8`** — Ampere Tensor Core warp instruction | 4× FP16 throughput vs FP32 SIMT |
| C2 | **`ldmatrix`** — load directly into tensor-core layout | Eliminates shared-mem→register transpose |
| C3 | **Double buffering** — ping-pong shmem between load and compute stages | Fully hides memory latency behind Tensor Core compute |

Expected improvement after Phase C: **4–16×** — reaching cuBLAS FP16 territory.

---

### Summary timeline

```
tiny-ton today    ~0.001 TF   (Level 3 at 128³)
  ↓  Phase A (2D grid + runtime loop + shmem A+B)
After Phase A     ~0.1–0.5 TF  (large sizes, FP32)
  ↓  Phase B (vectorized + swizzle + cp.async)
After Phase B     ~1–2 TF      (approaching cuBLAS FP32 target)
  ↓  Phase C (mma.sync + ldmatrix + double buffer)
After Phase C     ~6–12 TF     (matching cuBLAS FP16 on Ampere)

cuBLAS FP32 peak  ~2 TF   (Jetson Orin Nano, 4096³)
cuBLAS FP16 peak  ~12 TF  (with tensor cores)
```


In [ ]:
print('Done — no external handles to clean up (pure Python DSL).')
